# ATLOP + DREEAM — DocRED 학습 (Colab A100)

**최종 업데이트**: 2026-07-13: distant 사전학습(2만 문서, 1 epoch) → annotated fine-tune(전체 3,053 문서, 15 epoch) 2단계 학습에 **DREEAM 증거-유도 어텐션 손실(evidence-guided attention loss, evi_lambda=0.1)** 기법을 도입해 dev F1 / Ign F1을 산출하는 Colab 노트북.

## 구성
1. GPU 확인 (A100 기대)
2. 저장소 클론 + 의존성 설치
3. `Scripts/atlop/*.py` 최신 코드 덮어쓰기 (로컬 작업 중인 DREEAM 구현을 그대로 반영 — origin 브랜치에 아직 push되지 않았을 수 있어 노트북 자체에 내용을 박아 넣음)
4. DocRED 원본 데이터 다운로드 (HuggingFace `thunlp/docred` 저장소의 `.json.gz` 직접 다운로드 → 압축 해제) — git LFS 포인터 파일에 의존하지 않음
5. (선택) 파이프라인 정합성 스모크 테스트
6. **본 학습**: distant 20,000 문서 × 1 epoch 사전학습 → annotated 전체 × 15 epoch fine-tune, `--evi_lambda 0.1`(DREEAM)
7. 최종 dev F1 / Ign F1 출력
8. (선택) 결과물을 Google Drive에 백업

## 도입한 기법: DREEAM 증거-유도 어텐션 손실
Ma et al., "DREEAM: Guiding Attention with Evidence for Improving Document-Level Relation Extraction" (ACL 2023)의 아이디어를 재구현. ATLOP의 Localized Context Pooling이 만드는 head/tail 쌍별 문장 단위 attention 분포를, annotated/dev에 존재하는 정답 evidence 문장 쪽으로 supervise하는 보조 손실을 추가한다 (`Scripts/atlop/losses.py::evidence_loss`). `train_distant`는 evidence가 없어 자동으로 영향받지 않는다. `--evi_lambda 0.0`이면 기존 ATLOP 베이스라인과 완전히 동일하게 동작한다.


## 1. GPU 확인 (런타임 > 런타임 유형 변경 > GPU: A100 이어야 함)

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 2. 저장소 클론 + 의존성 설치

Colab 기본 torch(CUDA 빌드)는 그대로 두고, 코드가 필요로 하는 나머지 패키지만 버전 맞춰 설치한다.

In [ ]:
%cd /content
!rm -rf Information_Extraction
!git clone -b HY --single-branch https://github.com/multiful/Information_Extraction.git
%cd /content/Information_Extraction

!pip install -q transformers==4.57.6 tokenizers==0.22.2 accelerate==1.10.1 \
    datasets==4.5.0 huggingface_hub==0.36.2 tqdm==4.67.3


## 3. `Scripts/atlop/*.py` 최신 코드 반영

아래 5개 파일은 로컬에서 DREEAM 기능을 추가로 작업 중이라 origin의 `HY` 브랜치 커밋에는 아직 없을 수 있다. 클론 직후 이 코드로 덮어써서, push 여부와 무관하게 노트북이 항상 최신 구현으로 동작하도록 한다.

In [ ]:
%%writefile Scripts/atlop/losses.py
"""Adaptive Thresholding Loss (ATLoss) for ATLOP.

Re-implemented from the description in Zhou et al., "Document-Level Relation
Extraction with Adaptive Thresholding and Localized Context Pooling" (AAAI 2021)
and the reference repo wzhouad/ATLOP (losses.py). The repo's license is
unspecified, so this is a clean re-implementation, not a copy.

Class index 0 is the *threshold* class (TH) — it doubles as DocRED's "Na"
(no-relation) label, which lines up with `data.docred_io.build_rel2id`
(Na -> 0, the 96 real P-codes -> 1..96).

Idea: instead of a single global 0.5 threshold, the model learns a per-example
threshold as class 0's logit. A relation r is predicted iff logit_r > logit_TH.
The loss pushes every gold positive above TH and every negative below it, which
handles DocRED's ~97% NA imbalance structurally.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ATLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """logits, labels: (num_pairs, num_classes). labels is multi-hot with
        class 0 = Na/TH. Returns a scalar (mean over pairs)."""
        # TH label is always class 0.
        th_label = torch.zeros_like(labels, dtype=torch.float)
        th_label[:, 0] = 1.0
        labels = labels.clone().float()
        labels[:, 0] = 0.0  # class 0 is handled by TH, not as a real positive

        p_mask = labels + th_label   # positive relations + TH slot
        n_mask = 1 - labels          # everything that is NOT a gold positive

        # Part 1: rank each gold positive class above the TH class.
        #   mask out non-(positive/TH) logits so softmax is over {positives, TH}.
        logit1 = logits - (1 - p_mask) * 1e30
        loss1 = -(F.log_softmax(logit1, dim=-1) * labels).sum(1)

        # Part 2: rank the TH class above every negative class.
        #   mask out gold-positive logits so softmax is over {negatives, TH}.
        logit2 = logits - (1 - n_mask) * 1e30
        loss2 = -(F.log_softmax(logit2, dim=-1) * th_label).sum(1)

        loss = loss1 + loss2
        return loss.mean()

    def get_label(self, logits: torch.Tensor, num_labels: int = -1) -> torch.Tensor:
        """Turn logits into a multi-hot prediction (num_pairs, num_classes).

        A class is predicted iff its logit exceeds the TH (class 0) logit.
        `num_labels` optionally caps the number of positive relations per pair
        (top-k among those above threshold). If nothing beats TH, the pair is
        predicted as Na (class 0 set to 1)."""
        th_logit = logits[:, 0].unsqueeze(1)
        output = torch.zeros_like(logits)
        mask = logits > th_logit
        if num_labels > 0:
            top_v, _ = torch.topk(logits, num_labels, dim=1)
            top_v = top_v[:, -1]
            mask = (logits >= top_v.unsqueeze(1)) & mask
        output[mask] = 1.0
        # Rows with no relation above threshold -> predict Na (class 0).
        output[:, 0] = (output.sum(1) == 0).to(logits)
        return output


def evidence_loss(sent_attns: list[torch.Tensor], evidence: list[list[list[int]]]) -> torch.Tensor:
    """DREEAM-style evidence-guided attention loss.

    Re-implemented from the evidence-supervision idea in Ma et al., "DREEAM:
    Guiding Attention with Evidence for Improving Document-Level Relation
    Extraction" (ACL 2023): supervise the localized-context attention itself
    (not just the final relation logits) so it concentrates on the gold
    evidence sentences, rather than only on whichever tokens already scored
    highest.

    sent_attns[i]: (n_pair, n_sent) attention mass per sentence, for doc i's
        pairs in hts order (from DocREModel.get_hrt).
    evidence[i][k]: gold evidence sentence ids for pair k of doc i; empty for
        pairs with no evidence supervision (Na pairs, or splits without gold
        evidence like train_distant), which are skipped entirely.

    For each supervised pair, the target is a uniform distribution over its
    gold evidence sentences; the loss is the cross-entropy between that target
    and the model's per-sentence attention mass. Averaged over every
    supervised pair in the batch; 0.0 (no gradient) if none exist.
    """
    losses = []
    for attn, doc_evidence in zip(sent_attns, evidence):
        if attn.size(1) == 0:
            continue
        for k, evi in enumerate(doc_evidence):
            if not evi:
                continue
            target = attn.new_zeros(attn.size(1))
            target[evi] = 1.0 / len(evi)
            losses.append(-(target * torch.log(attn[k] + 1e-30)).sum())
    if not losses:
        return sent_attns[0].sum() * 0.0 if sent_attns else torch.tensor(0.0)
    return torch.stack(losses).mean()


In [ ]:
%%writefile Scripts/atlop/preprocess.py
"""ATLOP-style feature construction for DocRED.

Consumes raw documents from the team's shared loader
(`data.docred_dataset.DocREDataset`) — we do NOT re-implement data loading and
we reuse the shared `data.docred_io.build_rel2id` mapping, per PRD.md's
"모든 트랙이 DocREDataset을 그대로 입력으로" rule.

The only ATLOP-specific step is inserting a `"*"` marker token immediately
before and after every mention, then recording the subword index of each start
marker. ATLOP represents an entity by log-sum-exp pooling over its mentions'
start-marker hidden states, so these positions are what the model slices.
Re-implemented from wzhouad/ATLOP (prepro.py, read_docred); the repo's license
is unspecified so nothing is copied verbatim.

Marker positions here are recorded BEFORE the encoder's special tokens are
added; the model adds a `+offset` (1 for the leading [CLS]) when indexing.
"""

import sys
from pathlib import Path

from tqdm import tqdm
from transformers import PreTrainedTokenizerBase

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from data.docred_io import build_rel2id  # noqa: E402

NUM_CLASSES = 97  # Na/TH + 96 relation types
MARKER = "*"


def _encode_with_markers(doc: dict, tokenizer: PreTrainedTokenizerBase):
    """Tokenize a document word-by-word, wrapping each mention in `*` markers.

    Returns (input_ids, entity_pos, sent_pos) where entity_pos[e] is a list of
    (start, end) subword spans (start = the `*` start-marker index), one per
    mention, in vertexSet order, and sent_pos[s] is the (start, end) subword
    span of sentence s (same marker-token coordinate system, used to aggregate
    the localized-context attention into per-sentence mass for the DREEAM
    evidence-guided attention loss). Positions exclude the [CLS]/[SEP] added at
    the end (the model compensates with +offset).
    """
    sents = doc["sents"]
    vertex_set = doc["vertexSet"]

    # (sent_id, word_idx) sets where a `*` marker must be inserted.
    entity_start, entity_end = set(), set()
    for entity in vertex_set:
        for m in entity:
            sid = m["sent_id"]
            start_w, end_w = m["pos"]  # word-level [start, end) within the sentence
            entity_start.add((sid, start_w))
            entity_end.add((sid, end_w - 1))

    tokens: list[str] = []
    # sent_map[sid][word_idx] -> index in `tokens`; also holds len(sent) as a
    # sentinel so a mention ending on the last word maps cleanly.
    sent_map: list[dict[int, int]] = []
    for sid, sent in enumerate(sents):
        smap: dict[int, int] = {}
        for wi, word in enumerate(sent):
            wp = tokenizer.tokenize(word)
            if (sid, wi) in entity_start:
                wp = [MARKER] + wp
            if (sid, wi) in entity_end:
                wp = wp + [MARKER]
            smap[wi] = len(tokens)   # index of this word's first subword (or its `*` start marker)
            tokens.extend(wp)
        smap[len(sent)] = len(tokens)
        sent_map.append(smap)

    entity_pos: list[list[tuple[int, int]]] = []
    for entity in vertex_set:
        spans: list[tuple[int, int]] = []
        for m in entity:
            sid = m["sent_id"]
            start_w, end_w = m["pos"]
            start = sent_map[sid][start_w]   # the `*` start marker
            end = sent_map[sid][end_w]        # one past the `*` end marker
            spans.append((start, end))
        entity_pos.append(spans)

    sent_pos: list[tuple[int, int]] = [
        (sent_map[sid][0], sent_map[sid][len(sent)])
        for sid, sent in enumerate(sents)
    ]

    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    # Wrap with the encoder's leading/trailing special tokens ([CLS]..[SEP] /
    # <s>..</s>). The single leading token is why the model uses offset=1 when
    # indexing marker positions recorded above.
    input_ids = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]
    return input_ids, entity_pos, sent_pos


def build_features(
    docs,
    tokenizer: PreTrainedTokenizerBase,
    rel2id: dict[str, int] | None = None,
    show_progress: bool = True,
) -> list[dict]:
    """Turn an iterable of raw DocRED docs into ATLOP feature dicts.

    Each feature:
      input_ids   : list[int]  (with [CLS]/[SEP], `*` markers inserted)
      entity_pos  : list[list[(start, end)]]  marker-based spans per mention
      sent_pos    : list[(start, end)]        marker-based span per sentence
      hts         : list[(h, t)]              every ordered entity pair, h != t
      labels      : list[list[int]]           per pair, the positive class ids
                                              (Na pairs -> [0]); the collate_fn
                                              expands these to a (pairs, 97)
                                              multi-hot tensor. Stored sparsely
                                              so train_distant (100k docs) fits
                                              in memory.
      evidence    : list[list[int]]           per pair, gold evidence sentence
                                              ids (union across relations on
                                              that pair; [] if none — always []
                                              on train_distant, which carries no
                                              evidence). Used only for the
                                              DREEAM evidence-guided attention
                                              loss, ignored otherwise.
      title       : str
      num_entities: int
    """
    if rel2id is None:
        rel2id = build_rel2id()

    features: list[dict] = []
    it = tqdm(docs, desc="preprocess") if show_progress else docs
    for doc in it:
        input_ids, entity_pos, sent_pos = _encode_with_markers(doc, tokenizer)
        n_ent = len(doc["vertexSet"])

        # gold relations + evidence sentences keyed by (head_idx, tail_idx)
        triples: dict[tuple[int, int], list[int]] = {}
        pair_evidence: dict[tuple[int, int], set[int]] = {}
        for label in doc.get("labels", []):
            key = (label["h"], label["t"])
            triples.setdefault(key, []).append(rel2id[label["r"]])
            pair_evidence.setdefault(key, set()).update(label.get("evidence", []))

        hts, labels, evidence = [], [], []
        for h in range(n_ent):
            for t in range(n_ent):
                if h == t:
                    continue
                if (h, t) in triples:
                    pos = sorted(set(triples[(h, t)]))
                else:
                    pos = [0]  # Na / TH
                hts.append((h, t))
                labels.append(pos)
                evidence.append(sorted(pair_evidence.get((h, t), ())))

        features.append(
            {
                "input_ids": input_ids,
                "entity_pos": entity_pos,
                "sent_pos": sent_pos,
                "hts": hts,
                "labels": labels,
                "evidence": evidence,
                "title": doc["title"],
                "num_entities": n_ent,
            }
        )
    return features


In [ ]:
%%writefile Scripts/atlop/re_model.py
"""ATLOP relation-extraction model.

Re-implemented from wzhouad/ATLOP (model.py), described in Zhou et al. 2021.
The repo's license is unspecified, so this is a clean re-implementation with the
three ATLOP ingredients:

  1. Entity representation via log-sum-exp pooling over each mention's `*`
     start-marker hidden state (get_hrt).
  2. Localized Context Pooling: for a (head, tail) pair, multiply the two
     entities' token-attention distributions to focus a context vector on the
     tokens both attend to.
  3. Grouped bilinear classifier over [head; context] and [tail; context],
     trained with Adaptive Thresholding Loss (losses.ATLoss).

The encoder itself (BERT/RoBERTa/...) is passed in, so the same model works for
any HF encoder and can be handed a tiny random encoder for CPU smoke tests.
"""

import torch
import torch.nn as nn

from .losses import ATLoss, evidence_loss
from .long_input import process_long_input


class DocREModel(nn.Module):
    def __init__(self, config, encoder, emb_size: int = 768, block_size: int = 64,
                 num_labels: int = 97, offset: int = 1):
        super().__init__()
        self.config = config
        self.encoder = encoder
        self.hidden_size = config.hidden_size
        self.loss_fnt = ATLoss()

        self.head_extractor = nn.Linear(2 * config.hidden_size, emb_size)
        self.tail_extractor = nn.Linear(2 * config.hidden_size, emb_size)
        assert emb_size % block_size == 0, "emb_size must be divisible by block_size"
        self.bilinear = nn.Linear(emb_size * block_size, num_labels)

        self.emb_size = emb_size
        self.block_size = block_size
        self.num_labels = num_labels
        # +offset to skip the leading special token ([CLS]) when indexing markers.
        self.offset = offset

    def encode(self, input_ids, attention_mask):
        start_tokens = [self.config.cls_token_id]
        end_tokens = [self.config.sep_token_id]
        sequence_output, attention = process_long_input(
            self.encoder, input_ids, attention_mask, start_tokens, end_tokens
        )
        return sequence_output, attention

    def get_hrt(self, sequence_output, attention, entity_pos, hts, sent_pos=None):
        """Pool per-entity embeddings (log-sum-exp over mention markers) and
        per-pair localized-context vectors. Returns (hs, rs, ts), each
        (total_pairs, hidden), concatenated across the batch in hts order.

        If `sent_pos` is given (one (start, end) span per sentence, per doc),
        also returns `sent_attns`: a list of (n_pair_i, n_sent_i) tensors, one
        per doc, giving how much of each pair's localized-context attention
        mass falls on each sentence. This is the DREEAM evidence-guided
        attention signal — it lets the training loop compare "where the model
        is looking" against the gold evidence sentences, so low-scoring but
        evidence-relevant tokens aren't washed out just because the raw
        attention over them is small."""
        offset = self.offset
        _, num_heads, _, c = attention.size()
        hss, tss, rss = [], [], []
        sent_attns = [] if sent_pos is not None else None

        for i in range(len(entity_pos)):
            entity_embs, entity_atts = [], []
            for mentions in entity_pos[i]:
                if len(mentions) > 1:
                    m_emb, m_att = [], []
                    for start, _end in mentions:
                        if start + offset < c:
                            m_emb.append(sequence_output[i, start + offset])
                            m_att.append(attention[i, :, start + offset])
                    if m_emb:
                        e_emb = torch.logsumexp(torch.stack(m_emb, dim=0), dim=0)
                        e_att = torch.stack(m_att, dim=0).mean(0)
                    else:
                        e_emb = torch.zeros(self.hidden_size).to(sequence_output)
                        e_att = torch.zeros(num_heads, c).to(attention)
                else:
                    start, _end = mentions[0]
                    if start + offset < c:
                        e_emb = sequence_output[i, start + offset]
                        e_att = attention[i, :, start + offset]
                    else:
                        e_emb = torch.zeros(self.hidden_size).to(sequence_output)
                        e_att = torch.zeros(num_heads, c).to(attention)
                entity_embs.append(e_emb)
                entity_atts.append(e_att)

            entity_embs = torch.stack(entity_embs, dim=0)   # (n_ent, hidden)
            entity_atts = torch.stack(entity_atts, dim=0)   # (n_ent, heads, seq)

            # reshape(-1, 2) keeps shape (0, 2) for docs with no valid pairs.
            ht_i = torch.as_tensor(hts[i], dtype=torch.long,
                                   device=sequence_output.device).reshape(-1, 2)
            hs = torch.index_select(entity_embs, 0, ht_i[:, 0])
            ts = torch.index_select(entity_embs, 0, ht_i[:, 1])

            # Localized Context Pooling: element-wise product of head/tail
            # attention, averaged over heads, renormalized to a distribution.
            h_att = torch.index_select(entity_atts, 0, ht_i[:, 0])   # (n_pair, heads, seq)
            t_att = torch.index_select(entity_atts, 0, ht_i[:, 1])
            ht_att = (h_att * t_att).mean(1)                          # (n_pair, seq)
            ht_att = ht_att / (ht_att.sum(1, keepdim=True) + 1e-30)
            rs = torch.matmul(ht_att, sequence_output[i])            # (n_pair, hidden)

            hss.append(hs)
            tss.append(ts)
            rss.append(rs)

            if sent_pos is not None:
                spans = sent_pos[i]
                if spans and ht_att.size(0) > 0:
                    sent_attn = torch.stack(
                        [ht_att[:, min(s + offset, c): min(e + offset, c)].sum(dim=1)
                         for s, e in spans],
                        dim=1,
                    )  # (n_pair, n_sent)
                else:
                    sent_attn = ht_att.new_zeros((ht_att.size(0), 0))
                sent_attns.append(sent_attn)

        hss = torch.cat(hss, dim=0)
        tss = torch.cat(tss, dim=0)
        rss = torch.cat(rss, dim=0)
        if sent_pos is not None:
            return hss, rss, tss, sent_attns
        return hss, rss, tss

    def forward(self, input_ids, attention_mask, entity_pos, hts, labels=None,
                sent_pos=None, evidence=None, evi_lambda: float = 0.0):
        sequence_output, attention = self.encode(input_ids, attention_mask)
        if sent_pos is not None:
            hs, rs, ts, sent_attns = self.get_hrt(sequence_output, attention, entity_pos, hts, sent_pos)
        else:
            hs, rs, ts = self.get_hrt(sequence_output, attention, entity_pos, hts)
            sent_attns = None

        hs = torch.tanh(self.head_extractor(torch.cat([hs, rs], dim=1)))
        ts = torch.tanh(self.tail_extractor(torch.cat([ts, rs], dim=1)))

        # Grouped bilinear: split each vector into blocks and take outer products.
        b1 = hs.view(-1, self.emb_size // self.block_size, self.block_size)
        b2 = ts.view(-1, self.emb_size // self.block_size, self.block_size)
        bl = (b1.unsqueeze(3) * b2.unsqueeze(2)).view(-1, self.emb_size * self.block_size)
        logits = self.bilinear(bl)

        preds = self.loss_fnt.get_label(logits, num_labels=self.num_labels)
        output = (preds,)
        if labels is not None:
            # collate_fn hands us a dense (pairs, num_labels) float tensor.
            if not torch.is_tensor(labels):
                labels = torch.as_tensor(labels, dtype=torch.float)
            labels = labels.to(dtype=torch.float, device=logits.device)
            loss = self.loss_fnt(logits, labels)
            if sent_attns is not None and evidence is not None and evi_lambda > 0:
                loss = loss + evi_lambda * evidence_loss(sent_attns, evidence)
            output = (loss,) + output
        return output


In [ ]:
%%writefile Scripts/atlop/train_re.py
"""Train / evaluate the ATLOP baseline on DocRED (PRD track 2).

Two-stage flow; the stage order is selected with --distant_mode:
  pretrain (default — the original paper's order)
           stage 1  pretrain on noisy train_distant
           stage 2  fine-tune on clean train_annotated (hyperparams mirror
                    wzhouad/ATLOP's canonical DocRED config: bert-base-cased,
                    ATLoss, differential LRs)
  denoise  (team's earlier recipe, kept for comparison)
           stage 1  train on train_annotated — this model doubles as teacher
           stage 2  continue on train_distant AFTER dropping the distant
                    positive labels the teacher disagrees with
  none     train on train_annotated only
  eval     dev F1 / Ign F1 after every epoch via the shared Scripts.eval.scorer,
           predictions saved in the team's common format — directly comparable
           to the track-1 models.

--evi_lambda > 0 additionally enables the DREEAM evidence-guided attention loss
(Ma et al., ACL 2023): the localized-context attention is supervised against
gold evidence sentences (annotated/dev only; train_distant has no evidence, so
that stage is unaffected). Default 0.0 reproduces the pre-DREEAM baseline
exactly, since sent_pos/evidence are only forwarded to the model when enabled.

Run from the project root, e.g.

    # full run, paper order (GPU / Colab recommended — this repo's torch is CPU-only)
    python -m Scripts.atlop.train_re --epochs 30 --distant_limit 20000 --save_model

    # same, with DREEAM evidence-guided attention enabled
    python -m Scripts.atlop.train_re --epochs 30 --distant_limit 20000 --evi_lambda 0.1 --save_model

    # quick CPU sanity run on a handful of docs
    python -m Scripts.atlop.train_re --limit_docs 8 --epochs 1 --distant_epochs 1
"""

import argparse
import json
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModel, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from data.docred_dataset import DocREDataset          # noqa: E402
from data.docred_io import build_rel2id, NUM_CLASSES   # noqa: E402
from Scripts.atlop.preprocess import build_features     # noqa: E402
from Scripts.atlop.re_model import DocREModel           # noqa: E402
from Scripts.eval.scorer import evaluate                # noqa: E402

RESULTS_DIR = ROOT / "results"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_collate_fn(pad_token_id: int):
    """Pad input_ids to the batch max; keep entity_pos/hts ragged; expand the
    sparse positive-id label lists into one dense (total_pairs, NUM_CLASSES)
    multi-hot float tensor in hts order."""
    def collate(features: list[dict]) -> dict:
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids = torch.full((len(features), max_len), pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((len(features), max_len), dtype=torch.long)
        for i, f in enumerate(features):
            n = len(f["input_ids"])
            input_ids[i, :n] = torch.tensor(f["input_ids"], dtype=torch.long)
            attention_mask[i, :n] = 1

        pos_lists = [ids for f in features for ids in f["labels"]]
        labels = torch.zeros((len(pos_lists), NUM_CLASSES), dtype=torch.float)
        for i, ids in enumerate(pos_lists):
            labels[i, ids] = 1.0

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "entity_pos": [f["entity_pos"] for f in features],
            "hts": [f["hts"] for f in features],
            "sent_pos": [f["sent_pos"] for f in features],
            "evidence": [f["evidence"] for f in features],
            "labels": labels,
            "features": features,
        }
    return collate


@torch.no_grad()
def predict(model, loader, id2rel, device) -> list[dict]:
    """Run the model over a loader and emit predictions in the common format:
    [{"title","h_idx","t_idx","r"}], one row per predicted relation (r = P-code,
    class 0 / Na skipped)."""
    model.eval()
    out = []
    for batch in loader:
        preds = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            entity_pos=batch["entity_pos"],
            hts=batch["hts"],
        )[0].cpu().numpy()

        idx = 0
        for f in batch["features"]:
            n = len(f["hts"])
            doc_preds = preds[idx: idx + n]
            idx += n
            for (h, t), row in zip(f["hts"], doc_preds):
                for r in range(1, NUM_CLASSES):  # skip class 0 (Na/TH)
                    if row[r] == 1:
                        out.append({"title": f["title"], "h_idx": h, "t_idx": t, "r": id2rel[r]})
    return out


@torch.no_grad()
def denoise_features(model, loader, device) -> tuple[int, int]:
    """Filter distant-supervision labels with the current (annotated-trained)
    model acting as teacher. A distant positive label r on a pair is KEPT only
    if the teacher also ranks r above its adaptive threshold; every other
    positive is treated as distant noise and dropped (the pair falls back to
    Na). Mutates the underlying feature dicts in place (collate passes them
    through by reference). Returns (kept, dropped) positive-label counts."""
    model.eval()
    kept = dropped = 0
    for batch in loader:
        preds = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            entity_pos=batch["entity_pos"],
            hts=batch["hts"],
        )[0].cpu().numpy()
        idx = 0
        for f in batch["features"]:
            n = len(f["hts"])
            rows = preds[idx: idx + n]
            idx += n
            new_labels = []
            for ids, row in zip(f["labels"], rows):
                pos = [r for r in ids if r != 0]
                keep = [r for r in pos if row[r] == 1]
                kept += len(keep)
                dropped += len(pos) - len(keep)
                new_labels.append(keep if keep else [0])
            f["labels"] = new_labels
    return kept, dropped


def build_optim_sched(model, args, total_steps):
    """Fresh optimizer + linear-warmup scheduler. Built per stage so the distant
    pretrain and the annotated fine-tune each get their own schedule.
    Differential LR: pretrained encoder vs. freshly-initialized head."""
    new_layers = ("extractor", "bilinear")
    grouped = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in new_layers)],
         "lr": args.encoder_lr},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in new_layers)],
         "lr": args.classifier_lr},
    ]
    optimizer = torch.optim.AdamW(grouped, eps=1e-6)
    warmup_steps = int(total_steps * args.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    return optimizer, scheduler


def run_stage(model, loader, args, device, epochs, stage, dev_loader, dev_docs,
              ign_docs, id2rel):
    """Train `model` on `loader` for `epochs`, evaluating on dev after each epoch.
    Returns (last_metrics, last_predictions). `ign_docs` = train_annotated, used
    only for the Ign-F1 fact filter, regardless of which split we train on."""
    total_steps = max(1, len(loader) * epochs)
    optimizer, scheduler = build_optim_sched(model, args, total_steps)
    metrics, preds = None, None
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for step, batch in enumerate(loader):
            # sent_pos/evidence are only forwarded when the DREEAM evidence-guided
            # attention loss is enabled, so --evi_lambda 0 (the default) reproduces
            # the exact old code path/behavior.
            evi_kwargs = {}
            if args.evi_lambda > 0:
                evi_kwargs = {
                    "sent_pos": batch["sent_pos"],
                    "evidence": batch["evidence"],
                    "evi_lambda": args.evi_lambda,
                }
            loss = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                entity_pos=batch["entity_pos"],
                hts=batch["hts"],
                labels=batch["labels"],
                **evi_kwargs,
            )[0]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running += loss.item()
            if (step + 1) % args.log_every == 0:
                print(f"  [{stage}] epoch {epoch} step {step + 1}/{len(loader)} "
                      f"loss {running / (step + 1):.4f}")

        preds = predict(model, dev_loader, id2rel, device)
        metrics = evaluate(preds, dev_docs, ign_docs)
        print(f"[{stage} | epoch {epoch}] train_loss={running / max(1, len(loader)):.4f} "
              f"dev_F1={metrics['f1'] * 100:.2f} Ign_F1={metrics['ign_f1'] * 100:.2f} "
              f"(P={metrics['precision'] * 100:.2f} R={metrics['recall'] * 100:.2f})")
    return metrics, preds


def train(args):
    set_seed(args.seed)
    device = torch.device(args.device)
    print(f"[device] {device}")

    rel2id = build_rel2id()
    id2rel = {v: k for k, v in rel2id.items()}

    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
    collate = make_collate_fn(tokenizer.pad_token_id)

    # train_annotated (fine-tune stage) + dev (evaluation).
    train_docs = list(DocREDataset(args.train_split))
    dev_docs = list(DocREDataset(args.dev_split))
    if args.limit_docs > 0:
        train_docs = train_docs[: args.limit_docs]
        dev_docs = dev_docs[: args.limit_docs]
    print(f"[data] train={len(train_docs)} dev={len(dev_docs)} docs")

    dev_features = build_features(dev_docs, tokenizer, rel2id)
    dev_loader = DataLoader(dev_features, batch_size=args.eval_batch_size,
                            shuffle=False, collate_fn=collate)
    train_features = build_features(train_docs, tokenizer, rel2id)
    train_loader = DataLoader(train_features, batch_size=args.train_batch_size,
                              shuffle=True, collate_fn=collate)

    config = AutoConfig.from_pretrained(args.model_name_or_path, num_labels=NUM_CLASSES)
    # eager attention is required so the encoder returns attention weights
    # (localized context pooling needs them); sdpa/flash do not.
    encoder = AutoModel.from_pretrained(
        args.model_name_or_path, config=config, attn_implementation="eager"
    )
    config.cls_token_id = tokenizer.cls_token_id
    config.sep_token_id = tokenizer.sep_token_id
    model = DocREModel(config, encoder, emb_size=args.emb_size,
                       block_size=args.block_size, num_labels=NUM_CLASSES).to(device)

    def load_distant():
        cap = args.limit_docs if args.limit_docs > 0 else args.distant_limit
        docs = list(DocREDataset(args.distant_split))
        if cap > 0:
            docs = docs[:cap]
        return docs, build_features(docs, tokenizer, rel2id)

    if args.distant_mode == "pretrain":
        # Paper order: pretrain on the big noisy distant split first, then
        # fine-tune on the clean human-annotated split.
        distant_docs, distant_features = load_distant()
        print(f"[stage 1] distant pretrain on {len(distant_docs)} docs "
              f"({args.distant_epochs} epoch(s))")
        distant_loader = DataLoader(distant_features, batch_size=args.distant_batch_size,
                                    shuffle=True, collate_fn=collate)
        run_stage(model, distant_loader, args, device, args.distant_epochs,
                  "distant-pretrain", dev_loader, dev_docs, train_docs, id2rel)
        del distant_features, distant_loader, distant_docs

        print(f"[stage 2] annotated fine-tune on {len(train_docs)} docs ({args.epochs} epoch(s))")
        metrics, preds = run_stage(model, train_loader, args, device, args.epochs,
                                   "annotated-finetune", dev_loader, dev_docs, train_docs, id2rel)
    else:
        # Team recipe: supervised training on the clean annotated split first.
        # In denoise mode this model doubles as the teacher for stage 2.
        print(f"[stage 1] annotated train on {len(train_docs)} docs ({args.epochs} epoch(s))")
        metrics, preds = run_stage(model, train_loader, args, device, args.epochs,
                                   "annotated-train", dev_loader, dev_docs, train_docs, id2rel)

        # Optional stage 2: continue on train_distant AFTER stripping the
        # labels the stage-1 teacher disagrees with (self-training-style denoising).
        if args.distant_mode == "denoise":
            distant_docs, distant_features = load_distant()
            print(f"[stage 2] distant denoise+train on {len(distant_docs)} docs "
                  f"({args.distant_epochs} epoch(s))")
            denoise_loader = DataLoader(distant_features, batch_size=args.eval_batch_size,
                                        shuffle=False, collate_fn=collate)
            kept, dropped = denoise_features(model, denoise_loader, device)
            total = kept + dropped
            print(f"[stage 2] denoise: kept {kept}/{total} distant positive labels "
                  f"({dropped} dropped as noise)")
            distant_loader = DataLoader(distant_features, batch_size=args.distant_batch_size,
                                        shuffle=True, collate_fn=collate)
            metrics, preds = run_stage(model, distant_loader, args, device, args.distant_epochs,
                                       "distant-denoised", dev_loader, dev_docs, train_docs, id2rel)
            del distant_features, distant_loader, denoise_loader, distant_docs

    RESULTS_DIR.mkdir(exist_ok=True)
    pred_path = RESULTS_DIR / f"{args.run_name}_dev_predictions.json"
    with open(pred_path, "w", encoding="utf-8") as fp:
        json.dump(preds, fp, ensure_ascii=False)
    print(f"[saved] {pred_path}  ({len(preds)} predicted relations)")

    if args.save_model:
        ckpt = RESULTS_DIR / f"{args.run_name}.pt"
        torch.save(model.state_dict(), ckpt)
        print(f"[saved] {ckpt}")

    return metrics


def build_argparser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="ATLOP baseline on DocRED")
    p.add_argument("--model_name_or_path", default="bert-base-cased")
    p.add_argument("--train_split", default="train_annotated", help="stage-1 train split")
    p.add_argument("--dev_split", default="dev")
    p.add_argument("--run_name", default="atlop")
    p.add_argument("--epochs", type=int, default=30, help="annotated train/fine-tune epochs")
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--distant_mode", choices=["pretrain", "denoise", "none"], default="pretrain",
                   help="'pretrain' = paper order (distant pretrain -> annotated fine-tune), "
                        "'denoise' = team recipe (annotated train -> teacher-denoised distant), "
                        "'none' = annotated only")
    p.add_argument("--distant_split", default="train_distant")
    p.add_argument("--distant_epochs", type=int, default=1)
    p.add_argument("--distant_batch_size", type=int, default=4)
    p.add_argument("--distant_limit", type=int, default=0,
                   help="cap distant docs (0 = all 101,873; use e.g. 20000 to bound RAM/time on Colab)")
    p.add_argument("--encoder_lr", type=float, default=5e-5)
    p.add_argument("--classifier_lr", type=float, default=1e-4)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--emb_size", type=int, default=768)
    p.add_argument("--block_size", type=int, default=64)
    p.add_argument("--seed", type=int, default=66)
    p.add_argument("--evi_lambda", type=float, default=0.0,
                   help="weight of the DREEAM evidence-guided attention loss "
                        "(0 = off, reproduces the pre-DREEAM baseline exactly; "
                        "~0.1 enables it). No effect on splits without gold "
                        "evidence (e.g. train_distant), which stay Na/unsupervised.")
    p.add_argument("--limit_docs", type=int, default=0, help="cap train/dev docs (0 = all); for quick runs")
    p.add_argument("--log_every", type=int, default=50)
    p.add_argument("--save_model", action="store_true")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return p


if __name__ == "__main__":
    args = build_argparser().parse_args()
    train(args)


In [ ]:
%%writefile Scripts/atlop/smoke_test.py
"""End-to-end correctness check for the ATLOP pipeline on CPU.

This does NOT measure accuracy — it uses a tiny RANDOM-init BERT (no pretrained
download) on a handful of dev docs to prove every stage wires up correctly:
preprocessing + `*` markers -> encoder -> log-sum-exp entity pooling ->
localized context pooling -> bilinear classifier -> ATLoss (forward+backward) ->
get_label -> common prediction format -> shared scorer.

Run:  python -m Scripts.atlop.smoke_test
Real F1 numbers come from train_re.py with a real pretrained encoder on GPU.
"""

import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer, BertConfig, BertModel

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from data.docred_dataset import DocREDataset          # noqa: E402
from data.docred_io import build_rel2id, NUM_CLASSES   # noqa: E402
from Scripts.atlop.preprocess import build_features     # noqa: E402
from Scripts.atlop.re_model import DocREModel           # noqa: E402
from Scripts.atlop.train_re import make_collate_fn, predict  # noqa: E402
from Scripts.eval.scorer import evaluate                # noqa: E402

N_DOCS = 6
EMB_SIZE = 64
BLOCK_SIZE = 16


def main():
    torch.manual_seed(0)
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
    rel2id = build_rel2id()
    id2rel = {v: k for k, v in rel2id.items()}

    docs = [DocREDataset("dev")[i] for i in range(N_DOCS)]
    print(f"[data] {len(docs)} dev docs, entities per doc: "
          f"{[len(d['vertexSet']) for d in docs]}")

    features = build_features(docs, tokenizer, rel2id, show_progress=False)
    f0 = features[0]
    print(f"[preprocess] doc0: input_ids={len(f0['input_ids'])} "
          f"entities={f0['num_entities']} pairs={len(f0['hts'])} "
          f"labels(sparse pos-id lists, first 3)={f0['labels'][:3]}")
    # sanity: a marker '*' (id 115) should surround the first mention's first token
    star_id = tokenizer.convert_tokens_to_ids("*")
    first_start = f0["entity_pos"][0][0][0] + 1  # +1 for [CLS]
    assert f0["input_ids"][first_start] == star_id, "start marker not at recorded position"
    print(f"[preprocess] marker check OK (token at entity0/mention0 start is '*')")

    # tiny random encoder — offline, no pretrained weights
    config = BertConfig(
        vocab_size=tokenizer.vocab_size,
        hidden_size=EMB_SIZE, num_hidden_layers=2, num_attention_heads=4,
        intermediate_size=128, max_position_embeddings=1300,
        attn_implementation="eager",  # required for output_attentions=True
    )
    # add_pooling_layer=False: we only use last_hidden_state + attentions, so the
    # pooler would sit unused (and gradient-less). Dropping it keeps this check clean.
    encoder = BertModel(config, add_pooling_layer=False)
    config.cls_token_id = tokenizer.cls_token_id
    config.sep_token_id = tokenizer.sep_token_id
    model = DocREModel(config, encoder, emb_size=EMB_SIZE, block_size=BLOCK_SIZE,
                       num_labels=NUM_CLASSES)

    collate = make_collate_fn(tokenizer.pad_token_id)
    batch = collate(features[:3])

    # forward + loss + backward
    loss, preds = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        entity_pos=batch["entity_pos"],
        hts=batch["hts"],
        labels=batch["labels"],
    )
    print(f"[model] loss={loss.item():.4f}  preds shape={tuple(preds.shape)}")
    assert preds.shape[1] == NUM_CLASSES
    assert preds.shape[0] == sum(len(f["hts"]) for f in features[:3])
    loss.backward()
    grads = [p.grad is not None for p in model.parameters() if p.requires_grad]
    print(f"[model] backward OK, {sum(grads)}/{len(grads)} params have grads")
    assert all(grads), "some params did not receive gradients"

    # DREEAM evidence-guided attention: sent_pos/evidence wiring + backward.
    model.zero_grad()
    n_sent = [len(f["sent_pos"]) for f in features[:3]]
    n_evi_pairs = sum(1 for f in features[:3] for evi in f["evidence"] if evi)
    print(f"[dreeam] sentences per doc={n_sent}, pairs with gold evidence={n_evi_pairs}")
    loss_evi, preds_evi = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        entity_pos=batch["entity_pos"],
        hts=batch["hts"],
        labels=batch["labels"],
        sent_pos=batch["sent_pos"],
        evidence=batch["evidence"],
        evi_lambda=0.1,
    )
    # (predictions will differ slightly from the base run above due to encoder
    # dropout noise between forward passes, not the evidence loss itself)
    assert preds_evi.shape == preds.shape
    print(f"[dreeam] loss_with_evidence={loss_evi.item():.4f} (base={loss.item():.4f})")
    loss_evi.backward()
    grads_evi = [p.grad is not None for p in model.parameters() if p.requires_grad]
    print(f"[dreeam] backward OK, {sum(grads_evi)}/{len(grads_evi)} params have grads")
    assert all(grads_evi), "some params did not receive gradients with evidence loss enabled"

    # predict -> common format -> scorer
    from torch.utils.data import DataLoader
    loader = DataLoader(features, batch_size=2, shuffle=False, collate_fn=collate)
    predictions = predict(model, loader, id2rel, torch.device("cpu"))
    print(f"[predict] {len(predictions)} predicted relations, sample: "
          f"{predictions[0] if predictions else '(none — random weights)'}")

    metrics = evaluate(predictions, docs, docs)  # train_docs=docs just to exercise Ign path
    print(f"[scorer] F1={metrics['f1'] * 100:.2f} Ign_F1={metrics['ign_f1'] * 100:.2f} "
          f"P={metrics['precision'] * 100:.2f} R={metrics['recall'] * 100:.2f} "
          f"gold={metrics['num_gold']} submitted={metrics['num_submitted']}")

    print("\nSMOKE TEST PASSED - pipeline is wired end-to-end.")


if __name__ == "__main__":
    main()


## 4. DocRED 원본 데이터 준비

`docred_data/data/*.json`은 저장소에 Git LFS로 커밋되어 있어(용량 문제), LFS 스토리지 접근이 안 되면 클론만으로는 포인터 텍스트만 받게 된다. 대신 HuggingFace Hub의 원본 `thunlp/docred` 저장소에서 `.json.gz`를 직접 받아 압축 해제한다 (`data/docred_io.py`가 기대하는 형식과 동일한 원본 h/t/r 필드).

In [ ]:
import gzip, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_DIR = Path("/content/Information_Extraction/docred_data/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES = ["train_annotated", "train_distant", "dev", "test", "rel_info"]
for name in FILES:
    gz_path = hf_hub_download(repo_id="thunlp/docred", repo_type="dataset",
                               filename=f"data/{name}.json.gz")
    out_path = DATA_DIR / f"{name}.json"
    with gzip.open(gz_path, "rb") as f_in, open(out_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"[ok] {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")


## 5. (선택) 파이프라인 정합성 스모크 테스트

다운로드한 실제 인코더 없이 랜덤 소형 BERT로 전체 파이프라인(전처리 → 모델 → 손실 → DREEAM → 스코어러)이 잘 물려 있는지만 빠르게 확인. 정확도 수치가 아님.

In [ ]:
!python -m Scripts.atlop.smoke_test


## 6. 본 학습

- **Stage 1 (distant 사전학습)**: `train_distant` 중 20,000 문서, 1 epoch
- **Stage 2 (annotated fine-tune)**: `train_annotated` 전체 3,053 문서, 15 epoch
- **기법**: `--evi_lambda 0.1` → DREEAM 증거-유도 어텐션 손실 활성화
- 배치 크기는 A100 기준으로 논문 기본값(4)보다 올려 GPU를 더 활용하도록 설정(OOM 나면 낮출 것)
- 매 epoch마다 dev F1 / Ign F1 콘솔에 출력, 마지막 epoch(annotated 15번째)의 지표가 최종 결과

In [ ]:
import sys
sys.path.insert(0, "/content/Information_Extraction")

from Scripts.atlop.train_re import build_argparser, train

args = build_argparser().parse_args([
    "--distant_mode", "pretrain",       # 논문 순서: distant 사전학습 -> annotated fine-tune
    "--distant_limit", "20000",         # distant 20,000 문서만 샘플
    "--distant_epochs", "1",            # distant 1 epoch
    "--epochs", "15",                   # annotated 전체 15 epoch
    "--evi_lambda", "0.1",              # 기법: DREEAM 증거-유도 어텐션 손실
    "--train_batch_size", "8",
    "--distant_batch_size", "8",
    "--eval_batch_size", "16",
    "--run_name", "atlop_dreeam_colab",
    "--save_model",
])

metrics = train(args)

print("\n=== FINAL DEV METRICS (annotated fine-tune, epoch 15) ===")
print(f"F1         : {metrics['f1'] * 100:.2f}")
print(f"Ign F1     : {metrics['ign_f1'] * 100:.2f}")
print(f"Precision  : {metrics['precision'] * 100:.2f}")
print(f"Recall     : {metrics['recall'] * 100:.2f}")
print(f"gold={metrics['num_gold']} submitted={metrics['num_submitted']} correct={metrics['num_correct']}")


## 7. (선택) 결과물 Google Drive 백업

Colab 런타임이 끊기면 `/content` 산출물이 사라진다. 체크포인트(`results/atlop_dreeam_colab.pt`)와 예측(`results/atlop_dreeam_colab_dev_predictions.json`)을 보존하려면 아래 셀 실행.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path
dest = Path("/content/drive/MyDrive/atlop_dreeam_results")
dest.mkdir(parents=True, exist_ok=True)
for f in Path("/content/Information_Extraction/results").glob("atlop_dreeam_colab*"):
    shutil.copy(f, dest / f.name)
    print("copied", f.name)


## 참고: 베이스라인(기법 미적용) 대비 비교를 원한다면

`--evi_lambda 0.1` 대신 `--evi_lambda 0.0`(또는 인자 생략)으로 같은 셀을 다시 돌리면 DREEAM 미적용 순수 ATLOP 베이스라인 수치를 얻을 수 있다. `--run_name`을 다르게 지정해 결과 파일이 덮어써지지 않도록 할 것 (예: `atlop_baseline_colab`).